# Clasificador de Tendencia Suicida en Español con BERT (BETO)




Instalación (equivale a `!pip install bert-tensorflow`)

In [ ]:
!pip install transformers==4.40.0 accelerate

Importaciones

In [ ]:


from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

Configurar OUTPUT_DIR

In [ ]:

import os
OUTPUT_DIR = 'mejor_modelo_beto'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(' Model output directory: {} '.format(OUTPUT_DIR))

#Funciones de carga del dataset



In [ ]:
# Etiquetas del dataset
label_index = {'suicida': 1, 'no_suicida': 0}
index_label = {1: 'Suicida', 0: 'No Suicida'}

def load_dataset(path):

    df = pd.read_csv(path, engine='python')
    df['class'] = df['class'].map(label_index)
    return df

def download_and_load_datasets(path):

    df = load_dataset(path)

    train, late = train_test_split(df.values, random_state=42, test_size=0.25)
    val, test   = train_test_split(late,      random_state=42, test_size=0.01)

    train_df = pd.DataFrame(train, columns=['traducido', 'class'])
    val_df   = pd.DataFrame(val,   columns=['traducido', 'class'])
    test_df  = pd.DataFrame(test,  columns=['traducido', 'class'])

    for d in [train_df, val_df, test_df]:
        d['class'] = d['class'].astype(int)

    return train_df, val_df, test_df

 Cargar datasets


In [ ]:
DATA_PATH = '/content/data_raw.csv'

train, val, test = download_and_load_datasets(DATA_PATH)

# Usar solo el 30% del train para reducir tiempo de entrenamiento a ~5 min
#train = train.sample(frac=0.3, random_state=42).reset_index(drop=True)

print('Train:', len(train), '| Val:', len(val), '| Test:', len(test))
train.head()

 Definir columnas y lista de etiquetas



In [ ]:
DATA_COLUMN  = 'traducido'
LABEL_COLUMN = 'class'
label_list   = [0, 1]

train.columns

# Convertir a InputExamples


In [ ]:
class InputExample:

    def __init__(self, guid, text_a, text_b=None, label=None):
        self.guid   = guid
        self.text_a = text_a
        self.text_b = text_b
        self.label  = label


train_InputExamples = train.apply(
    lambda x: InputExample(guid=None, text_a=x[DATA_COLUMN], text_b=None, label=x[LABEL_COLUMN]),
    axis=1
)

test_InputExamples = test.apply(
    lambda x: InputExample(guid=None, text_a=x[DATA_COLUMN], text_b=None, label=x[LABEL_COLUMN]),
    axis=1
)

In [ ]:
BERT_MODEL_HUB = 'dccuchile/bert-base-spanish-wwm-cased'

def create_tokenizer_from_hub_module():

    return BertTokenizerFast.from_pretrained(BERT_MODEL_HUB)


tokenizer = create_tokenizer_from_hub_module()

Probar tokenizador


In [ ]:
tokenizer.tokenize("No quiero seguir viviendo, todo es un dolor constante")

## Convertir ejemplos a features



In [ ]:
class InputFeatures:

    def __init__(self, input_ids, input_mask, segment_ids, label_id):
        self.input_ids   = input_ids
        self.input_mask  = input_mask
        self.segment_ids = segment_ids
        self.label_id    = label_id


def convert_examples_to_features(examples, label_list, max_seq_length, tokenizer):

    features = []
    for example in examples:
        encoding = tokenizer(
            str(example.text_a),
            truncation=True,
            padding='max_length',
            max_length=max_seq_length
        )
        features.append(InputFeatures(
            input_ids   = encoding['input_ids'],
            input_mask  = encoding['attention_mask'],
            segment_ids = encoding.get('token_type_ids', [0] * max_seq_length),
            label_id    = int(example.label)
        ))
    return features


MAX_SEQ_LENGTH = 32   # 128→32: textos cortos bastan para detectar tendencia suicida

train_features = convert_examples_to_features(train_InputExamples, label_list, MAX_SEQ_LENGTH, tokenizer)
test_features  = convert_examples_to_features(test_InputExamples,  label_list, MAX_SEQ_LENGTH, tokenizer)

print(f'Train features: {len(train_features)}')
print(f'Test  features: {len(test_features)}')

## Crear modelo


In [ ]:
def create_model(num_labels):

    model = BertForSequenceClassification.from_pretrained(
        BERT_MODEL_HUB,
        num_labels=num_labels,
        ignore_mismatched_sizes=True
    )
    return model

In [ ]:
def model_fn_builder(num_labels, learning_rate, num_train_steps, num_warmup_steps):

    model = create_model(num_labels=num_labels)


    optimizer = AdamW(model.parameters(), lr=learning_rate, eps=1e-8)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_train_steps
    )

    return model, optimizer, scheduler

 Hiperparámetros


In [ ]:
BATCH_SIZE              = 128   # máximo que cabe en T4 con seq_length=32
LEARNING_RATE           = 2e-5
NUM_TRAIN_EPOCHS        = 1.0   # 1 sola época
WARMUP_PROPORTION       = 0.1
SAVE_CHECKPOINTS_STEPS  = 500
SAVE_SUMMARY_STEPS      = 100

In [ ]:
num_train_steps  = int(len(train_features) / BATCH_SIZE * NUM_TRAIN_EPOCHS)
num_warmup_steps = int(num_train_steps * WARMUP_PROPORTION)

run_config



In [ ]:


run_config = {
    'model_dir':              OUTPUT_DIR,
    'save_summary_steps':     SAVE_SUMMARY_STEPS,
    'save_checkpoints_steps': SAVE_CHECKPOINTS_STEPS,
    'device':                 torch.device('cuda' if torch.cuda.is_available() else 'cpu')
}

print('Device:', run_config['device'])

In [ ]:
class BertEstimator:

    def __init__(self, model, optimizer, scheduler, config, params):
        self.model     = model.to(config['device'])
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.config    = config
        self.params    = params
        self.device    = config['device']
        # Scaler para mixed precision (fp16) — solo activo si hay GPU
        self.scaler    = torch.cuda.amp.GradScaler(enabled=self.device.type == 'cuda')

    def train(self, input_fn, max_steps):

        self.model.train()
        loader     = input_fn()
        steps_done = 0

        while steps_done < max_steps:
            for batch in loader:
                if steps_done >= max_steps:
                    break
                input_ids, input_mask, segment_ids, label_ids = [b.to(self.device) for b in batch]
                self.optimizer.zero_grad()

                with torch.cuda.amp.autocast(enabled=self.device.type == 'cuda'):
                    outputs = self.model(
                        input_ids=input_ids,
                        attention_mask=input_mask,
                        token_type_ids=segment_ids,
                        labels=label_ids
                    )

                self.scaler.scale(outputs.loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.scheduler.step()
                steps_done += 1

                if steps_done % self.config['save_summary_steps'] == 0:
                    print(f'  Step {steps_done}/{max_steps} — loss: {outputs.loss.item():.4f}')

    def evaluate(self, input_fn, steps=None):

        self.model.eval()
        loader     = input_fn()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for i, batch in enumerate(loader):
                if steps and i >= steps:
                    break
                input_ids, input_mask, segment_ids, label_ids = [b.to(self.device) for b in batch]
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=input_mask,
                    token_type_ids=segment_ids
                )
                preds = outputs.logits.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(label_ids.cpu().numpy())

        results = {
            'eval_accuracy': accuracy_score(all_labels, all_preds),
            'f1_score':      f1_score(all_labels, all_preds, average='weighted'),
        }
        return results

    def predict(self, input_fn):

        self.model.eval()
        loader = input_fn()
        all_probs, all_labels_pred = [], []

        with torch.no_grad():
            for batch in loader:
                input_ids, input_mask, segment_ids, _ = [b.to(self.device) for b in batch]
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=input_mask,
                    token_type_ids=segment_ids
                )
                probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
                preds = np.argmax(probs, axis=1)
                all_probs.extend(probs)
                all_labels_pred.extend(preds)

        return [{'probabilities': p, 'labels': l} for p, l in zip(all_probs, all_labels_pred)]


# Construir modelo y estimador
model_fn, optimizer, scheduler = model_fn_builder(
    num_labels=len(label_list),
    learning_rate=LEARNING_RATE,
    num_train_steps=num_train_steps,
    num_warmup_steps=num_warmup_steps
)

estimator = BertEstimator(
    model=model_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    config=run_config,
    params={'batch_size': BATCH_SIZE}
)

##input_fn_builder para train


In [ ]:
def input_fn_builder(features, seq_length, is_training, drop_remainder=False):

    input_ids   = torch.tensor([f.input_ids   for f in features], dtype=torch.long)
    input_mask  = torch.tensor([f.input_mask  for f in features], dtype=torch.long)
    segment_ids = torch.tensor([f.segment_ids for f in features], dtype=torch.long)
    label_ids   = torch.tensor([f.label_id    for f in features], dtype=torch.long)

    dataset = TensorDataset(input_ids, input_mask, segment_ids, label_ids)

    def input_fn():
        return DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=is_training,
            drop_last=drop_remainder
        )

    return input_fn


train_input_fn = input_fn_builder(
    features=train_features,
    seq_length=MAX_SEQ_LENGTH,
    is_training=True,
    drop_remainder=False
)

## Entrenar el modelo


In [ ]:
print(f'Beginning Training!')
current_time = datetime.now()

estimator.train(input_fn=train_input_fn, max_steps=num_train_steps)

print('Tiempo de entrenamiento ', datetime.now() - current_time)

Beginning Training!


In [ ]:
test_input_fn = input_fn_builder(
    features=test_features,
    seq_length=MAX_SEQ_LENGTH,
    is_training=False,
    drop_remainder=False
)

## Evaluar el modelo


In [ ]:
estimator.evaluate(input_fn=test_input_fn, steps=None)

In [ ]:
def getPrediction(in_sentences):
    labels = ['No Suicida', 'Suicida']

    input_examples = [
        InputExample(guid='', text_a=x, text_b=None, label=0)
        for x in in_sentences
    ]

    input_features = convert_examples_to_features(
        input_examples, label_list, MAX_SEQ_LENGTH, tokenizer
    )

    predict_input_fn = input_fn_builder(
        features=input_features,
        seq_length=MAX_SEQ_LENGTH,
        is_training=False,
        drop_remainder=False
    )

    predictions = estimator.predict(predict_input_fn)

    return [
        (sentence, prediction['probabilities'], labels[prediction['labels']])
        for sentence, prediction in zip(in_sentences, predictions)
    ]

## Frases de prueba



In [ ]:
pred_sentences = [
    "Ya no quiero seguir viviendo, todo es un dolor constante",
    "Hoy fue un día difícil pero mañana será mejor",
    "Siento que soy una carga para todos y sería mejor desaparecer",
    "Estoy aprendiendo a manejar mis emociones con ayuda profesional",
    "No encuentro ninguna razón para continuar"
]

In [ ]:
predicciones = getPrediction(pred_sentences)
predicciones